<a href="https://colab.research.google.com/github/jayneamol/e-realestate/blob/master/agric_advosory_asr_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The ASR steps include:


1.   HuggingFace dataset access through audio ingestion.
2.  Dialect-aware preprocessing.
3. Whisper fine-tuning and incremental training using the AfriVoices KE Kikuyu (754h (190h untranscribed), 215GB) and Kalenjin (521h (16h untranscribed), 144GB) datasets.



SALT = github.com/SunbirdAI/salt  
LEB = github.com/jqug/leb

Model: Whisper large-v2 fine-tuning, starting from Sunbird/asr-whisper-large-v2-salt

In [ ]:
import os, re, sys, argparse, random, unicodedata, logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional
from collections import defaultdict

In [ ]:
# disable HF visualisation

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

In [ ]:
# install and environment validation

import numpy as np
import torch
import torchaudio
import torchaudio.transforms as TAT
import evaluate
from datasets import load_dataset, Audio, concatenate_datasets, load_from_disk
from transformers import (
    WhisperFeatureExtractor,
    WhisperProcessor,
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
# constants and paths

SALT_MODEL_ID  = "Sunbird/asr-whisper-large-v2-salt"   # [SALT] warm-start base
BASE_MODEL_ID  = "openai/whisper-large-v2"             # fallback if SALT gated

In [ ]:
# dataset IDs

KIK_DATASET_ID = "Anv-ke/kikuyu"
KAL_DATASET_ID = "Anv-ke/Kalenjin"

In [ ]:
# file paths

CACHE_DIR      = Path(os.environ.get("HF_DATASETS_CACHE", "/data/hf_cache"))
KIK_CACHE      = Path("/data/kik_processed")
KAL_CACHE      = Path("/data/kal_processed")
KIK_OUT_DIR    = Path("./checkpoints/whisper-salt-kik")
KAL_OUT_DIR    = Path("./checkpoints/whisper-salt-kal-lora")

In [ ]:
# audio contraits (splitting into 30s clips for Whisper, downgradding to 16kHZ for practical phone conversations)

TARGET_SR      = 16_000      # Whisper requires 16kHz
MIN_DURATION   = 0.5         # seconds — discard sub-500ms clips
MAX_DURATION   = 30.0        # seconds — Whisper's hard input limit
MIN_CHARS      = 3           # discard near-empty transcripts

In [ ]:
# language tokens in SALT (must find a way to swap with Kalenjin and Kikuyu tokens)

SALT_LANG_TOKEN_IDS = {
    "eng": 50259,   # English (Ugandan accent)
    "ach": 50357,   # Acholi
    "lgg": 50356,   # Lugbara
    "lug": 50355,   # Luganda
    "nyn": 50354,   # Runyankole
    "teo": 50353,   # Ateso
}

In [ ]:
# custom tokens for Kikuyu and Kalenjin

NEW_SPECIAL_TOKENS = [
    "<|kik|>", "<|kln|>",                                          # language
    "[kabete]", "[mathira]", "[muranga]", "[ndia]", "[gichugu]",   # Kikuyu dialects
    "[nandi]", "[kipsigis]",                                       # Kalenjin dialects
]

In [ ]:
# HF token authentication

def step_01_authenticate() -> None:

    from huggingface_hub import login, whoami

    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False)
        log.info("Logged in via HF_TOKEN env variable")
    else:
        # attempt cached login (set by huggingface-cli login)

        try:
            info = whoami()
            log.info(f"Already authenticated as: {info['name']}")
        except Exception:
            raise RuntimeError(
                "Not authenticated. Set HF_TOKEN env variable or run "
                "`huggingface-cli login` first."
            )

    # verify access to both gated datasets

    from huggingface_hub import dataset_info
    for ds_id in [KIK_DATASET_ID, KAL_DATASET_ID]:
        try:
            info = dataset_info(ds_id)
            log.info(f"  ✓ Access confirmed: {ds_id} (gated={info.gated})")
        except Exception as e:
            raise RuntimeError(
                f"Cannot access {ds_id}. Visit huggingface.co/datasets/{ds_id} "
                f"and accept the data-sharing terms. Error: {e}"
            )

In [ ]:
# data streaming from HF
# sampling to view schema

def step_02_stream_and_inspect(dataset_id: str, n_examples: int = 3) -> None:
    """
    AfriVoices-KE schema:
      - 'actualSentence'    (str)  : transcript for SCRIPTED utterances
      - 'transcript'        (str)  : transcript for UNSCRIPTED utterances
      - 'type'              (bool) : True = scripted, False/None = unscripted
      - 'sentenceDialect'   (str)  : dialect label
      - 'domain'            (str)  : topic domain (Healthcare, Agriculture, …)
      - 'audio'             (dict) : {'array': np.ndarray, 'sampling_rate': int}
      - 'mediaPathId'       (str)  : unique file identifier
      - 'recorder_uuid'     (str)  : speaker ID
    """
    log.info(f"\n{'='*60}")
    log.info(f"STEP 02  Streaming inspection: {dataset_id}")
    log.info(f"{'='*60}")

    stream = load_dataset(
        dataset_id,
        split="train",
        streaming=True,
        trust_remote_code=True,
    )

    log.info(f"Features: {stream.features}")

    for i, ex in enumerate(stream.take(n_examples)):
        audio   = ex["audio"]
        dur     = len(audio["array"]) / audio["sampling_rate"]
        text    = ex.get("actualSentence") if ex.get("type") else ex.get("transcript")
        dialect = ex.get("sentenceDialect", "unknown")
        domain  = ex.get("domain", "unknown")

        log.info(
            f"\n  Example {i+1}:"
            f"\n    Duration  : {dur:.2f}s  @ {audio['sampling_rate']}Hz"
            f"\n    Scripted  : {ex.get('type')}"
            f"\n    Dialect   : {dialect}"
            f"\n    Domain    : {domain}"
            f"\n    Text      : {text[:80] if text else '(no transcript)'}..."
        )


In [ ]:
# data streaming from HF
# loading full dataset

def step_02_load_full(dataset_id: str) -> Any:
    """
    Load the full dataset and cast audio to 16kHz.
    Downloads to CACHE_DIR; uses num_proc=8 for parallel shard fetching.

    Note on disk space:
      kikuyu:   ~215 GB raw + ~90 GB preprocessed mel cache
      Kalenjin: ~144 GB raw + ~62 GB preprocessed mel cache
    """
    log.info(f"\n{'='*60}")
    log.info(f"STEP 02  Loading full dataset: {dataset_id}")
    log.info(f"{'='*60}")

    ds = load_dataset(
        dataset_id,
        trust_remote_code=True,
        num_proc=8,
        cache_dir=str(CACHE_DIR),
    )

    # cast audio column to 16kHz — torchaudio polyphase resampler under the hood

    ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

    log.info(f"Loaded: {ds}")
    for split, subset in ds.items():
        log.info(f"  {split}: {len(subset):,} examples")

    return ds


In [ ]:
# unifying labels for scripted + unscripted (both have different schema)

def step_03_get_label(example: dict) -> str:
    """
    Unified label extraction for AfriVoices-KE dual-column schema.

    AfriVoices-KE stores transcripts in two different columns depending
    on whether the utterance was scripted or spontaneous:
      - scripted   → example['actualSentence']
      - unscripted → example['transcript']
    The 'type' column (bool) flags which applies.

    ~32% of Kikuyu unscripted and ~5% of Kalenjin unscripted
    currently have no transcript.
    """
    if example.get("type"):   # True = scripted
        text = example.get("actualSentence") or ""
    else:                     # False / None = unscripted
        text = example.get("transcript") or ""
    return text.strip()


In [ ]:
# text normalisation (both from SALT process but restoring Kikuyu diacritics stripped by clean_text())

def _nfc(text: str) -> str:
    """[SALT clean_text] Unicode NFC normalisation."""
    return unicodedata.normalize("NFC", text)


def _strip_diacritics(text: str, preserve: tuple = ()) -> str:
    """
    [SALT clean_text strip_diacritics=True] NFD decompose → remove Mn category.
    preserve: tuple of composed characters to protect before decomposition.

    Example: preserve=("ũ","ĩ") protects Kikuyu tilde characters.
    """

    # replace preserved chars with placeholders

    placeholders = {}
    for i, ch in enumerate(preserve):
        ph = f"\x00PRESERVE{i}\x00"
        text = text.replace(ch, ph)
        placeholders[ph] = ch

    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = unicodedata.normalize("NFC", text)

    # restore preserved chars

    for ph, ch in placeholders.items():
        text = text.replace(ph, ch)
    return text


In [ ]:
# normalise Kikuyu

def step_04_normalise_kikuyu(text: str) -> str:
    """
    [SALT clean_text + CUSTOM]
    Normalise Kikuyu (kik) text for ASR training.

    Kikuyu orthography notes:
      ũ (U+0169) and ĩ (U+0129) are PHONEMIC — must NOT be stripped.
      Vowel doubling (aa, ii, uu) marks length — preserve.
      Apostrophe used in some dialects (n'akolera style) — preserve.

    LEB equivalent (when stable):
      from leb.preprocessing import clean_text
      clean_text(example, "source", lower=True,
                 unicode_normalise="NFC", remove_punctuation=True,
                 preserve_chars=["ũ","ĩ","'"])
    """
    text = _nfc(text)
    text = text.lower()

    # remove punctuation except apostrophe and Kikuyu-specific chars
    # [SALT clean_text remove_punctuation=True]
    text = re.sub(r"""[!?,;:\.\"""«»\(\)\[\]{}—–\-]""", "", text)

    # Strip non-Kikuyu Latin diacritics only (NOT ũ or ĩ)
    # [CUSTOM — LEB's default would strip all, including ũ/ĩ]
    text = re.sub(r"[àáâäæãåā]", "a", text)
    text = re.sub(r"[èéêëēė]",   "e", text)
    text = re.sub(r"[òóôöõøō]",  "o", text)

    # Collapse whitespace [SALT clean_text]
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
# normalise Kalenjin

def step_04_normalise_kalenjin(text: str) -> str:
    """
    [SALT clean_text]
    Normalise Kalenjin (kln) text — Nandi + Kipsigis dialects.

    Kalenjin notes:
      Tone marks (acute accent) are NOT part of standard written Nandi/Kipsigis
      → strip them (same as LEB's strip_diacritics=True).
      Apostrophes mark ejective consonants (k', t', ch') — PRESERVE.
      Length-marking double vowels (aa, ee) — preserve.

    LEB equivalent:
      clean_text(example, "source", lower=True, strip_diacritics=True,
                 preserve_chars=["'"])
    """
    text = _nfc(text)
    text = text.lower()

    # strip ALL diacritics (tone marks not standard in written Kalenjin)
    # [SALT clean_text strip_diacritics=True]
    text = _strip_diacritics(text, preserve=())   # no chars to preserve here

    # remove punctuation, keep apostrophe for ejectives
    # [SALT clean_text remove_punctuation=True, CUSTOM preserve_chars=["'"]]
    text = re.sub(r"""[!?,;:\.\"""«»\(\)\[\]{}—–\-]""", "", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

    def step_04_apply_dialect_prefix(
    text: str,
    dialect: str,
    lang: str,
) -> str:
    """
    [CUSTOM — no SALT equivalent]
    Prepend dialect token so the decoder learns dialect-conditioned transcription.
    The tokenizer treats these as regular tokens (added in step_08_setup_model).

    Examples:
      "[kabete] mũndũ agĩtwarwo na ndege"  (Gĩ-Kabete Kikuyu)
      "[nandi]  che ng'atunot ak"           (Nandi Kalenjin)
    """
    d_map = KIK_DIALECT_MAP if lang == "kik" else KLN_DIALECT_MAP
    prefix = d_map.get(dialect, "")
    return f"{prefix} {text}".strip() if prefix else text


In [ ]:
# audio augmentation

def step_04_apply_dialect_prefix(
    text: str,
    dialect: str,
    lang: str,
) -> str:

    d_map = KIK_DIALECT_MAP if lang == "kik" else KLN_DIALECT_MAP
    prefix = d_map.get(dialect, "")
    return f"{prefix} {text}".strip() if prefix else text

    if random.random() > p:
        return audio_array

    wav    = torch.tensor(audio_array).unsqueeze(0)          # (1, T)
    down   = TAT.Resample(sr, 8_000)(wav)                    # → 8kHz
    up     = TAT.Resample(8_000, sr)(down)                   # → 16kHz again
    return up.squeeze(0).numpy().astype(np.float32)


def step_05_spec_augment(mel: torch.Tensor) -> torch.Tensor:

 mel = mel.unsqueeze(0)                                   # (1, 80, 3000)
    mel = TAT.FrequencyMasking(freq_mask_param=27)(mel)
    mel = TAT.TimeMasking(time_mask_param=100, p=0.05)(mel)
    mel = TAT.TimeMasking(time_mask_param=100, p=0.05)(mel)  # twice
    return mel.squeeze(0)                                    # (80, 3000)

In [ ]:
# preprocessing map

def step_07_build_preprocess_fn(
    processor: WhisperProcessor,
    lang: str,
    apply_augmentation: bool = True,
) -> callable:
    """
      get_label()
      normalise_*()
      noise + phone aug
      feature_extractor()
    """
    normalise_fn = (
        step_04_normalise_kikuyu if lang == "kik"
        else step_04_normalise_kalenjin
    )
    dialect_map = KIK_DIALECT_MAP if lang == "kik" else KLN_DIALECT_MAP

    def preprocess(example: dict) -> dict:

        # get raw audio

        audio_array = example["audio"]["array"].astype(np.float32)
        sr          = example["audio"]["sampling_rate"]
        assert sr == TARGET_SR, f"Expected 16kHz, got {sr}Hz"

        # quality gate

        duration = len(audio_array) / sr
        raw_text = step_03_get_label(example)
        if not (MIN_DURATION <= duration <= MAX_DURATION):
            return {}   # will be filtered by remove_empty later
        if len(raw_text) < MIN_CHARS:
            return {}

        # audio augmentation (train only)

        if apply_augmentation:
            audio_array = step_05_noise_augmentation(audio_array, p=0.5)
            audio_array = step_05_phone_simulation(audio_array, sr, p=0.3)

        # log-mel feature extraction (80, 3000)

        input_features = processor.feature_extractor(
            audio_array,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
        ).input_features[0]            # shape: (80, 3000)

        # text normalise + dialect prefix + tokenise
        norm_text    = normalise_fn(raw_text)
        dialect      = example.get("sentenceDialect", "")
        prefixed     = step_04_apply_dialect_prefix(norm_text, dialect, lang)
        label_ids    = processor.tokenizer(prefixed).input_ids

        # retain metadata for stratified eval
        return {
            "input_features": input_features,        # (80, 3000) float32
            "labels":         label_ids,             # List[int]
            "duration":       duration,
            "domain":         example.get("domain", "unknown"),
            "dialect":        dialect or "unknown",
        }

    return preprocess



In [ ]:
# whisper preprocessor

def step_07_run_preprocessing(
    raw_ds: Any,
    processor: WhisperProcessor,
    lang: str,
    cache_path: Path,
    force_reprocess: bool = False,
) -> Any:
    """
    Apply preprocessing to all splits and save to disk.
    Loading from cache on subsequent runs avoids re-processing 215GB+.
    """
    log.info(f"\n{'='*60}")
    log.info(f"STEP 07  Preprocessing: {lang.upper()}")
    log.info(f"{'='*60}")

    if cache_path.exists() and not force_reprocess:
        log.info(f"  Loading from cache: {cache_path}")
        return load_from_disk(str(cache_path))

    # Build per-split preprocessing functions
    # (augmentation only on train split)
    processed_splits = {}
    for split_name, split_ds in raw_ds.items():
        is_train = "train" in split_name
        preprocess_fn = step_07_build_preprocess_fn(
            processor, lang, apply_augmentation=is_train
        )
        log.info(f"  Processing {split_name} ({len(split_ds):,} examples)...")

        processed = split_ds.map(
            preprocess_fn,
            remove_columns=split_ds.column_names,
            num_proc=8,
            desc=f"{lang} {split_name}",
            batched=False,
        )

        # Remove empty dicts (failed quality gate)
        processed = processed.filter(
            lambda ex: len(ex["labels"]) > 0,
            num_proc=4,
        )
        log.info(f"    → {len(processed):,} valid examples after filtering")
        processed_splits[split_name] = processed

    from datasets import DatasetDict
    ds_dict = DatasetDict(processed_splits)
    ds_dict.save_to_disk(str(cache_path))
    log.info(f"  Saved to: {cache_path}")
    return ds_dict

In [ ]:
# model tokeniser

def step_08_setup_model_and_processor() -> tuple[WhisperForConditionalGeneration, WhisperProcessor]:

    log.info(f"\n{'='*60}")
    log.info(f"STEP 08  Loading SALT model + processor")
    log.info(f"{'='*60}")

    try:
        processor = WhisperProcessor.from_pretrained(
            SALT_MODEL_ID, task="transcribe"
        )
        model = WhisperForConditionalGeneration.from_pretrained(SALT_MODEL_ID)
        log.info(f"  ✓ Loaded SALT checkpoint: {SALT_MODEL_ID}")
    except Exception as e:
        log.warning(f"  Could not load SALT model ({e}). Falling back to base Whisper.")
        processor = WhisperProcessor.from_pretrained(
            BASE_MODEL_ID, language="sw", task="transcribe"
        )
        model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID)
        log.info(f"  ⚠ Loaded fallback: {BASE_MODEL_ID}")

    # [CUSTOM] Add AfriVoices-KE language + dialect tokens
    n_added = processor.tokenizer.add_tokens(NEW_SPECIAL_TOKENS)
    log.info(f"  Added {n_added} new tokens → vocab size: {len(processor.tokenizer)}")

    # Record new token IDs for forced decoding
    kik_token_id = processor.tokenizer.convert_tokens_to_ids("<|kik|>")
    kln_token_id = processor.tokenizer.convert_tokens_to_ids("<|kln|>")
    log.info(f"  kik token id: {kik_token_id}")
    log.info(f"  kln token id: {kln_token_id}")

    # Resize model embeddings to cover new tokens
    model.resize_token_embeddings(len(processor.tokenizer))

    # Whisper fine-tuning config — disable caching (incompatible with grad ckpt)
    model.config.use_cache          = False
    model.config.suppress_tokens    = []
    model.config.forced_decoder_ids = None

    return model, processor, kik_token_id, kln_token_id

In [ ]:
#  data collator

@dataclass
class SaltDataCollator:

  processor:              WhisperProcessor
    decoder_start_token_id: int
    apply_spec_augment:     bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # audio features: already (80, 3000)
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_feats, return_tensors="pt"
        )  # → (B, 80, 3000)

        # SpecAugment on mel (train only)
        if self.apply_spec_augment:
            augmented = []
            for i in range(batch["input_features"].shape[0]):
                augmented.append(step_05_spec_augment(batch["input_features"][i]))
            batch["input_features"] = torch.stack(augmented)

        # labels: pad to max sequence length in batch
        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_feats, return_tensors="pt"
        )
        # replace padding token id (usually 50256) with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # strip leading decoder_start_token_id if present (Whisper adds it)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [ ]:
# evaluation using WER

def step_10_build_compute_metrics(processor: WhisperProcessor) -> callable:
    """
    [standard evaluate library]
    Returns a compute_metrics function for Seq2SeqTrainer.
    Computes WER on the decoded token ids vs. reference label ids.
    """
    wer_metric = evaluate.load("wer")

    def compute_metrics(pred) -> Dict[str, float]:
        pred_ids  = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

        wer = 100 * wer_metric.compute(
            predictions=pred_str, references=label_str
        )
        return {"wer": round(wer, 2)}

    return compute_metrics


In [ ]:
# whisper large v2 finetuning decoder only

def step_11_finetune_phase1(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    processed_ds: Any,
    lang: str,
    lang_token_id: int,
    output_dir: Path,
) -> WhisperForConditionalGeneration:


    log.info(f"\n{'='*60}")
    log.info(f"STEP 11  Phase 1 fine-tune ({lang.upper()}) — decoder only")
    log.info(f"{'='*60}")

    # [SALT pattern] Freeze encoder to preserve East African acoustic knowledge
    model.model.encoder.requires_grad_(False)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"  Trainable params: {trainable/1e6:.0f}M / {sum(p.numel() for p in model.parameters())/1e6:.0f}M total")

    # [CUSTOM] Set forced decoder language token (using SALT integer ID pattern)
    lang_token_str = processor.tokenizer.decode([lang_token_id])
    model.config.forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(
        language=lang_token_str, task="transcribe"
    )

    p1_dir = output_dir / "phase1"
    collator = SaltDataCollator(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
        apply_spec_augment=True,
    )

    args = Seq2SeqTrainingArguments(
        output_dir=str(p1_dir),
        # Batch & accumulation
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,   # effective batch = 16
        # LR — lower than cold start because SALT gives us a warm start
        learning_rate=3e-5,
        warmup_steps=200,
        max_steps=10_000,               # ~33% fewer than cold-start (was 15k)
        # Memory
        gradient_checkpointing=True,
        fp16=True,
        # Evaluation
        evaluation_strategy="steps",
        eval_steps=1_000,
        save_steps=1_000,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        # Generation
        predict_with_generate=True,
        generation_max_length=448,
        # Misc
        dataloader_num_workers=4,
        remove_unused_columns=False,
        report_to=["tensorboard"],
        push_to_hub=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=processed_ds["train"],
        eval_dataset=processed_ds.get("dev", processed_ds.get("validation")),
        data_collator=collator,
        compute_metrics=step_10_build_compute_metrics(processor),
        tokenizer=processor.feature_extractor,
    )

    log.info("  Starting Phase 1 training...")
    trainer.train()

    best_path = p1_dir / "best"
    trainer.save_model(str(best_path))
    log.info(f"  Phase 1 best model saved to: {best_path}")
    return model

In [ ]:
# whisper large v2 finetuning full model

def step_12_finetune_phase2(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    processed_ds: Any,
    output_dir: Path,
) -> WhisperForConditionalGeneration:


    log.info(f"\n{'='*60}")
    log.info(f"STEP 12  Phase 2 fine-tune — full model (encoder + decoder)")
    log.info(f"{'='*60}")

    # Unfreeze encoder
    model.model.encoder.requires_grad_(True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"  Trainable params: {trainable/1e6:.0f}M (full model)")

    p2_dir = output_dir / "phase2"
    collator = SaltDataCollator(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
        apply_spec_augment=True,
    )

    args = Seq2SeqTrainingArguments(
        output_dir=str(p2_dir),
        per_device_train_batch_size=4,   # halved vs phase 1 — more VRAM
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 16
        learning_rate=1e-5,              # 3× lower — don't destroy SALT weights
        warmup_steps=100,
        max_steps=5_000,
        gradient_checkpointing=True,
        fp16=True,
        evaluation_strategy="steps",
        eval_steps=500,
        save_steps=500,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=448,
        dataloader_num_workers=4,
        remove_unused_columns=False,
        report_to=["tensorboard"],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=processed_ds["train"],
        eval_dataset=processed_ds.get("dev", processed_ds.get("validation")),
        data_collator=collator,
        compute_metrics=step_10_build_compute_metrics(processor),
        tokenizer=processor.feature_extractor,
    )

    log.info("  Starting Phase 2 training...")
    trainer.train()

    best_path = p2_dir / "best"
    trainer.save_model(str(best_path))
    log.info(f"  Phase 2 best model saved to: {best_path}")
    return model


In [ ]:
#LoRA incremental training

def step_13_lora_incremental_kalenjin(
    kik_checkpoint: Path,
    processor: WhisperProcessor,
    kal_processed: Any,
    kik_processed: Any,
    kln_token_id: int,
    output_dir: Path,
) -> None:

    log.info(f"\n{'='*60}")
    log.info(f"STEP 13  LoRA incremental training: Kikuyu → Kalenjin")
    log.info(f"{'='*60}")

    # load the fine-tuned Kikuyu checkpoint
    base = WhisperForConditionalGeneration.from_pretrained(str(kik_checkpoint))
    base.resize_token_embeddings(len(processor.tokenizer))
    base.config.use_cache          = False
    base.config.suppress_tokens    = []
    base.config.forced_decoder_ids = None

    # LoRA adapters
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=32,             # rank — higher = more adaptation capacity
        lora_alpha=64,    # scaling factor: alpha/r = 2.0
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "out_proj", "fc1", "fc2"],
        bias="none",
        modules_to_save=["embed_tokens"],  # fully train embedding layer (new tokens)
    )
    lora_model = get_peft_model(base, lora_cfg)

    trainable, total = 0, 0
    for p in lora_model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    log.info(f"  LoRA trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)")

    # set Kalenjin language token
    kln_token_str = processor.tokenizer.decode([kln_token_id])
    lora_model.config.forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(
        language=kln_token_str, task="transcribe"
    )

    # 20% of kal_train size, stratified across Kikuyu dialects
    kal_train_size = len(kal_processed["train"])
    replay_n       = int(kal_train_size * 0.20)
    kik_replay     = (
        kik_processed["train"]
        .shuffle(seed=42)
        .select(range(min(replay_n, len(kik_processed["train"]))))
    )
    combined_train = concatenate_datasets([
        kal_processed["train"], kik_replay
    ]).shuffle(seed=42)

    log.info(
        f"  Kalenjin train: {kal_train_size:,} | "
        f"Kikuyu replay: {len(kik_replay):,} | "
        f"Combined: {len(combined_train):,}"
    )

    # collator — no SpecAugment on LoRA run (noise aug already applied)
    collator = SaltDataCollator(
        processor=processor,
        decoder_start_token_id=lora_model.config.decoder_start_token_id,
        apply_spec_augment=False,
    )

    lora_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=1,
        learning_rate=1e-4,
        warmup_steps=200,
        max_steps=4_000,
        gradient_checkpointing=True,
        fp16=True,
        evaluation_strategy="steps",
        eval_steps=500,
        save_steps=500,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=448,
        dataloader_num_workers=4,
        remove_unused_columns=False,
        report_to=["tensorboard"],
    )

    trainer = Seq2SeqTrainer(
        model=lora_model,
        args=lora_args,
        train_dataset=combined_train,
        eval_dataset=kal_processed.get("dev", kal_processed.get("validation")),
        data_collator=collator,
        compute_metrics=step_10_build_compute_metrics(processor),
        tokenizer=processor.feature_extractor,
    )

    log.info("  Starting LoRA incremental training...")
    trainer.train()

    # save LoRA adapter (~60 MB)
    adapter_path = output_dir / "kal-lora-adapter"
    lora_model.save_pretrained(str(adapter_path))
    log.info(f"  LoRA adapter saved to: {adapter_path}")

    # evaluate Kikuyu retention
    log.info("\n  Evaluating Kikuyu WER retention after Kalenjin adaptation...")
    kik_results = trainer.evaluate(
        eval_dataset=kik_processed.get("dev", kik_processed.get("validation"))
    )
    log.info(f"  Kikuyu WER after LoRA adaptation: {kik_results.get('eval_wer', '?')}%")
    log.info(f"  (If >5pp higher than phase 2 baseline, reduce LoRA r or increase replay %)")


In [ ]:
# evaluating by dialect

def step_14_evaluate_by_dialect(
    model_or_adapter_path: Path,
    base_checkpoint: Optional[Path],
    processor: WhisperProcessor,
    processed_ds: Any,
    lang: str,
    lang_token_id: int,
    split: str = "dev_test",
) -> None:


    log.info(f"\n{'='*60}")
    log.info(f"STEP 14  Dialect-stratified evaluation: {lang.upper()} ({split})")
    log.info(f"{'='*60}")

    # load model (merge LoRA if adapter path given)

    if base_checkpoint and (model_or_adapter_path / "adapter_config.json").exists():
        base = WhisperForConditionalGeneration.from_pretrained(str(base_checkpoint))
        base.resize_token_embeddings(len(processor.tokenizer))
        peft_model = PeftModel.from_pretrained(base, str(model_or_adapter_path))
        model = peft_model.merge_and_unload()
        log.info("  Loaded LoRA adapter + merged into base")
    else:
        model = WhisperForConditionalGeneration.from_pretrained(str(model_or_adapter_path))
        model.resize_token_embeddings(len(processor.tokenizer))
        log.info("  Loaded full model checkpoint")

    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device)

    # forced language token
    lang_str = processor.tokenizer.decode([lang_token_id])
    forced_ids = processor.tokenizer.get_decoder_prompt_ids(
        language=lang_str, task="transcribe"
    )

    wer_metric = evaluate.load("wer")
    eval_split = processed_ds.get(split, processed_ds.get("dev"))
    results_by_dialect = defaultdict(lambda: {"preds": [], "refs": []})
    results_by_domain  = defaultdict(lambda: {"preds": [], "refs": []})

    for ex in eval_split:
        inp = torch.tensor(ex["input_features"]).unsqueeze(0).to(device)

        with torch.no_grad():
            ids = model.generate(
                inp,
                forced_decoder_ids=forced_ids,
                max_new_tokens=448,
                num_beams=5,
            )

        pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
        ref  = processor.tokenizer.decode(ex["labels"], skip_special_tokens=True)

        results_by_dialect[ex.get("dialect", "unknown")]["preds"].append(pred)
        results_by_dialect[ex.get("dialect", "unknown")]["refs"].append(ref)
        results_by_domain[ex.get("domain", "unknown")]["preds"].append(pred)
        results_by_domain[ex.get("domain", "unknown")]["refs"].append(ref)

    log.info(f"\n  ── WER by dialect ──")
    for grp, data in sorted(results_by_dialect.items()):
        w = 100 * wer_metric.compute(predictions=data["preds"], references=data["refs"])
        log.info(f"    {grp:<22}  WER = {w:5.1f}%  (n={len(data['preds'])})")

    log.info(f"\n  ── WER by domain ──")
    for grp, data in sorted(results_by_domain.items()):
        w = 100 * wer_metric.compute(predictions=data["preds"], references=data["refs"])
        log.info(f"    {grp:<22}  WER = {w:5.1f}%  (n={len(data['preds'])})")

In [ ]:
# entrypoint

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="AfriVoices-KE × SALT ASR Pipeline")
    p.add_argument(
        "--language", choices=["kik", "kln", "both"], default="both",
        help="Which language to train (kik=Kikuyu, kln=Kalenjin incremental, both=full pipeline)"
    )
    p.add_argument("--inspect-only",  action="store_true", help="Only stream & inspect, no training")
    p.add_argument("--force-reprocess", action="store_true", help="Re-run preprocessing even if cache exists")
    p.add_argument("--skip-phase2",   action="store_true", help="Skip Phase 2 (encoder unfrozen) — faster, higher WER")
    p.add_argument("--eval-split",    default="dev_test",  help="Split to use for final eval")
    return p.parse_args()


def main() -> None:
    args = parse_args()

    # auth
    step_01_authenticate()

    # stream inspect

    if args.language in ("kik", "both"):
        step_02_stream_and_inspect(KIK_DATASET_ID)
    if args.language in ("kln", "both"):
        step_02_stream_and_inspect(KAL_DATASET_ID)

    if args.inspect_only:
        log.info("--inspect-only set, stopping after inspection.")
        return

    # model + processor (needed before preprocessing for tokenizer)

    model, processor, kik_token_id, kln_token_id = step_08_setup_model_and_processor()

    # full load

    kik_raw, kal_raw = None, None
    if args.language in ("kik", "both"):
        kik_raw = step_02_load_full(KIK_DATASET_ID)
    if args.language in ("kln", "both"):
        kal_raw = step_02_load_full(KAL_DATASET_ID)

    # preprocess + cache

    kik_processed, kal_processed = None, None
    if kik_raw is not None:
        kik_processed = step_07_run_preprocessing(
            kik_raw, processor, "kik", KIK_CACHE,
            force_reprocess=args.force_reprocess
        )
    if kal_raw is not None:
        kal_processed = step_07_run_preprocessing(
            kal_raw, processor, "kln", KAL_CACHE,
            force_reprocess=args.force_reprocess
        )

    # Kikuyu fine-tune

    if kik_processed is not None:
        model = step_11_finetune_phase1(
            model, processor, kik_processed,
            lang="kik", lang_token_id=kik_token_id,
            output_dir=KIK_OUT_DIR
        )
        if not args.skip_phase2:
            model = step_12_finetune_phase2(
                model, processor, kik_processed,
                output_dir=KIK_OUT_DIR
            )
        kik_final_path = KIK_OUT_DIR / ("phase2/best" if not args.skip_phase2 else "phase1/best")

        # Evaluate Kikuyu
        step_14_evaluate_by_dialect(
            kik_final_path, None, processor,
            kik_processed, "kik", kik_token_id,
            split=args.eval_split
        )

    # Kalenjin LoRA incremental
    if kal_processed is not None and kik_processed is not None:
        kik_base = KIK_OUT_DIR / ("phase2/best" if not args.skip_phase2 else "phase1/best")
        step_13_lora_incremental_kalenjin(
            kik_checkpoint=kik_base,
            processor=processor,
            kal_processed=kal_processed,
            kik_processed=kik_processed,
            kln_token_id=kln_token_id,
            output_dir=KAL_OUT_DIR,
        )

        # Evaluate Kalenjin
        adapter_path = KAL_OUT_DIR / "kal-lora-adapter"
        step_14_evaluate_by_dialect(
            adapter_path, kik_base, processor,
            kal_processed, "kln", kln_token_id,
            split=args.eval_split
        )

    log.info("\n✓ Pipeline complete.")


if __name__ == "__main__":
    main()